# Switch to a local model

So far our reviews went to a model in a data centre. **Open-weight models** can be downloaded and run on your own computer: no data leaves the machine, and there is no price per token. Ollama runs such models behind the same kind of API, so our code hardly changes.

In this notebook, we will:
1. Start Ollama and connect the client to a small local model.
2. Classify the same 10 reviews locally and with `gpt-6-luna`.
3. Compare accuracy, speed, cost and privacy.

**Time:** about 15 minutes, self-study. Model execution and the one-time model download add waiting time. Reasoning effort was already explored in notebook 02; here the local model answers without thinking.

**Self-check:** allow about 5 additional minutes for the questions at the end of the core exercise. Optional extensions take extra time.

## Libraries and settings

In [ ]:
# Libraries
import os
import time
import platform
from datetime import datetime
from platform import python_version

import pandas as pd
from openai import OpenAI
from sklearn.metrics import accuracy_score

# Course helpers
from llm_course import DATA, ROOT, ask, parse_label

# API model and course price assumptions in USD per 1 million tokens
MODEL = 'gpt-6-luna'
PRICE_INPUT_PER_M = 0.10
PRICE_OUTPUT_PER_M = 0.50

# Local model: small enough for a codespace without a graphics card
LOCAL_MODEL = 'qwen3:1.7b'

# Check that the API key is available (never print the key itself)
print('API key found:', bool(os.environ.get('OPENAI_API_KEY')))

# Show current working directory
print(os.getcwd())

## Step 1: Start Ollama and download the model

The script installs Ollama if it is missing, starts it and downloads the model (about 1.4 GB, only the first time).

In [ ]:
!bash {ROOT}/scripts/start_ollama.sh {LOCAL_MODEL}

### Connect the client

Ollama listens on this computer at `localhost:11434`. We use the same OpenAI library as before with a local address (`base_url`) and model name. Ollama ignores the API key, but the library wants one.

In [ ]:
# A client that talks to Ollama on this computer
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

def ask_local(prompt):
    """Send one prompt to the local model and return the answer text."""
    r = local.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        reasoning_effort='none',
    )
    return r.choices[0].message.content

## Step 2: Classify 10 reviews with both models

We use 10 reviews with silver labels from notebook 03. Both models get the same prompt. Ten reviews keep the waiting time short: without a graphics card, the local model needs several seconds for a long review.

In [ ]:
# Reviews with silver labels from notebook 03
silver_path = DATA / 'processed/silver_labels.csv'
assert silver_path.exists(), 'Run notebook 03 first: it creates the silver labels.'
silver = pd.read_csv(silver_path)

# 10 random reviews
test = silver.sample(10, random_state=42).reset_index(drop=True)

PROMPT = ('Classify the sentiment of this customer review. '
          'Answer with exactly one word: positive, negative or mixed.\n\nReview: ')

test[['id', 'stars', 'silver', 'text']].head(5)

### Local model

`parse_label()` accepts a single label (with optional final punctuation) or a final `Overall: label` answer. It ignores `<think>...</think>` blocks and never guesses from words in an explanation. Other answers become `'unparsed'` and count as errors.

In [ ]:
# Local model: one review after the other, and we stop the time
labels = []
start = time.perf_counter()

for text in test['text']:
    answer = ask_local(PROMPT + text)
    labels.append(parse_label(answer))

local_seconds = (time.perf_counter() - start) / len(test)
test['local'] = labels

print('Seconds per review:', round(local_seconds, 2))

### API model

`use_cache=False` sends every request again instead of using a saved answer. We need this to measure the real speed. **Every run of this cell incurs API charges.** Their size depends on the texts and generated output tokens. The calculation below estimates them using the course prices.

In [ ]:
# API model: the same reviews, and we add up the tokens
labels = []
input_tokens = 0
output_tokens = 0
start = time.perf_counter()

for text in test['text']:
    r = ask(model=MODEL, input=PROMPT + text, reasoning={'effort': 'low'}, use_cache=False)
    labels.append(parse_label(r.output_text))
    input_tokens += r.usage.input_tokens
    output_tokens += r.usage.output_tokens

api_seconds = (time.perf_counter() - start) / len(test)
test['api'] = labels

# Cost: tokens / 1 million x price per 1 million tokens
api_cost = (input_tokens * PRICE_INPUT_PER_M + output_tokens * PRICE_OUTPUT_PER_M) / 1_000_000

print('Seconds per review:', round(api_seconds, 2))
print('Input tokens:', input_tokens, '| output tokens:', output_tokens)

## Step 3: Compare accuracy, speed and cost

In [ ]:
comparison = pd.DataFrame(
    {
        LOCAL_MODEL: [accuracy_score(test['silver'], test['local']), local_seconds, 0.0],
        MODEL: [accuracy_score(test['silver'], test['api']), api_seconds, api_cost],
    },
    index=['accuracy (silver labels)', 'seconds per review', f'API cost in USD ({len(test)} reviews)'],
)
comparison.round(4)

In [ ]:
# Where do the two models disagree with the silver labels?
pd.set_option('display.max_colwidth', 300)
errors = test[(test['local'] != test['silver']) | (test['api'] != test['silver'])]
errors[['id', 'silver', 'local', 'api', 'text']]

## Interpret the comparison

- **Accuracy:** read the measured scores and inspect the disagreements below the table. Which model makes more errors in this sample? Do not assume the winner from the model size.
- **Fair comparison?** The silver labels were made by `gpt-6-luna` itself. A model agrees with its own labels more easily; a fair comparison needs gold labels made by people.
- **10 reviews are few:** one additional correct or incorrect prediction changes the accuracy by 10 percentage points. For a real decision you would test many more.
- **Cost:** the local model has no price per token, but it is not free: it needs hardware, electricity and somebody who runs it.
- **Privacy:** with a local model, the reviews never leave the computer. That can be the deciding argument.

## Check your understanding

Take about 5 minutes. Answer in one or two sentences per question, using the outputs above where useful. No additional model calls are needed. Try first, then open the hints.

1. What changes when the client points to localhost instead of the external API? Where are the review texts processed?
2. Use your comparison table: which model is more accurate and which is faster? Can these results establish that one model is always better?
3. Why can silver labels from the API model favour that model in the comparison? What would make the reference more independent?
4. Why do we bypass the response cache for timing? Why do zero API costs not mean that a local model is free to operate?
5. Give one requirement favouring local processing and one favouring an API. Before choosing either, which simpler baseline could you test for this task?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The base URL and model change. Ollama serves the local model on this computer; the external API processes the submitted text at the provider.

2. Use the measured values, even if they differ from expectations. Ten reviews and one machine are weak evidence for a general claim.

3. The API model may reproduce its own reference labels more easily. Independently created human gold labels provide a fairer reference.

4. Cached replies measure cache access rather than model inference. Local operation still needs hardware, electricity and maintenance.

5. For example: a requirement to keep texts on the machine favours local processing; limited hardware or low measured latency may favour an API. Test TF-IDF with a classifier as a baseline.

</details>

## Optional: test a larger sample

If you have time, change `sample(10, ...)` to `sample(30, ...)` in step 2 and rerun the comparison. Are the same conclusions supported? This takes longer and sends new API requests. Other extensions are unnecessary for the core comparison.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')